# Carga de datos del Reto 1 en Kafka y HDFS

El flujo es **archivos locales → productor Kafka → tópico → consumidor Kafka → HDFS**. Los archivos se envían en fragmentos binarios para admitir CSV, JSON, HTML y XLSX sin exceder el tamaño habitual de un mensaje Kafka.

Se suben únicamente los archivos seleccionados como originales en [ESTRUCTURA_HDFS_RETO1.md](../../ESTRUCTURA_HDFS_RETO1.md). La estructura remota queda bajo `originales/`; incluye la fotografía procesada de Etxebide como solicitaste y `pisos_euskadi.csv` en `originales/pisos/`. No se crean carpetas `processed` ni `reports`.

## 1. Dependencias

Ejecuta esta celda una vez en el kernel del notebook. El entorno también necesita acceso de red al broker Kafka y al servicio WebHDFS.

In [1]:
%pip install confluent-kafka hdfs

Note: you may need to restart the kernel to use updated packages.


## 2. Conexión y configuración

Configura el broker Kafka, el endpoint WebHDFS, el usuario y el tópico asignado al proyecto Reto 1. Los valores se pueden cambiar aquí o sobrescribir con las variables de entorno `KAFKA_BROKER`, `HDFS_URL`, `HDFS_USER`, `KAFKA_TOPIC` y `HDFS_ROOT`. El tópico debe estar creado en Kafka y el usuario debe tener permisos de productor y consumidor.

In [2]:
import base64
import csv
import json
import math
import os
import socket
import tempfile
import time
import uuid
from pathlib import Path, PurePosixPath

from confluent_kafka import Consumer, KafkaError, KafkaException, Producer
from hdfs import InsecureClient

USUARIO = os.getenv("HDFS_USER", "iabd-user11")
KAFKA_BROKER = os.getenv("KAFKA_BROKER", "10.22.87.215:9092")
HDFS_URL = os.getenv("HDFS_URL", "http://10.22.87.215:50070")
HDFS_ROOT = PurePosixPath(os.getenv("HDFS_ROOT", f"/user/{USUARIO}/reto1"))
RUN_ID = uuid.uuid4().hex
TOPIC = os.getenv("KAFKA_TOPIC", f"reto1_archivos_{USUARIO}")
GROUP_ID = f"grupo_reto1_{USUARIO}_{RUN_ID}"
CHUNK_SIZE = 128 * 1024
IDLE_TIMEOUT_SECONDS = 600

if not HDFS_ROOT.is_absolute() or ".." in HDFS_ROOT.parts:
    raise ValueError("HDFS_ROOT debe ser una ruta HDFS absoluta sin '..'.")

project_candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in project_candidates
     if (path / "DatosIker").is_dir() and (path / "DatosAlvaro").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encuentra la raíz del proyecto. Abre Jupyter desde la carpeta "
        "Reto1 o ajusta PROJECT_ROOT manualmente."
    )

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Kafka: {KAFKA_BROKER} | tópico: {TOPIC}")
print(f"WebHDFS: {HDFS_URL} | destino: {HDFS_ROOT}")
print(f"Usuario HDFS: {USUARIO}")

Proyecto: c:\Users\Alvaro.Morcillo\Desktop\Reto1
Kafka: 10.22.87.215:9092 | tópico: reto1_archivos_iabd-user11
WebHDFS: http://10.22.87.215:50070 | destino: /user/iabd-user11/reto1
Usuario HDFS: iabd-user11


## 3. Selección y validación de archivos

Se valida que los archivos seleccionados existan y no estén vacíos; también se comprueba que los JSON se puedan leer y que los CSV tengan encabezado. Los HTML con `notFound` en su nombre o contenido se omiten. No se incluyen conjuntos procesados de Fotocasa/EMAL, informes, scripts ni backups. Los CSV limpios de Etxebide se guardan bajo `originales/etxebide` por decisión del proyecto.

In [3]:
files = []

def add_file(local_relative, hdfs_relative):
    files.append({
        "local_path": PROJECT_ROOT / local_relative,
        "hdfs_relative": PurePosixPath(hdfs_relative).as_posix(),
    })

add_file(
    Path("DatosIker") / "originales" / "fotocasa" / "MunicipiosCAE_2025_19_02.xlsx",
    "originales/fotocasa/MunicipiosCAE_2025_19_02.xlsx",
)
add_file(
    Path("DatosIker") / "originales" / "EMAL.-Barrios-Municipios.-2016-2026_es.xlsx",
    "originales/emal/EMAL.-Barrios-Municipios.-2016-2026_es.xlsx",
)

eustat_originales = (
    "eustat_distribucion_renta_municipal.json",
    "eustat_distribucion_renta_por_edad.json",
    "eustat_indicadores_municipales.json",
    "eustat_percentiles_renta_municipales.json",
    "eustat_renta_por_edad.json",
    "eustat_renta_por_municipio.json",
    "metadatos_distribucion_renta_municipal.json",
    "metadatos_distribucion_renta_por_edad.json",
    "metadatos_indicadores_municipales.json",
    "metadatos_percentiles_renta_municipales.json",
    "metadatos_renta_por_edad.json",
    "metadatos_renta_por_municipio.json",
)
for filename in eustat_originales:
    add_file(Path("DatosIker") / "originales" / filename, f"originales/eustat/{filename}")

fotocasa_originales = PROJECT_ROOT / "DatosIker" / "originales" / "fotocasa"
if not fotocasa_originales.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de originales: {fotocasa_originales}")

html_files = sorted(fotocasa_originales.rglob("*.html"))
excluded_html = []
for path in html_files:
    if "notfound" in path.name.lower() or b"notfound" in path.read_bytes().lower():
        excluded_html.append(path)
        continue
    relative_html_path = path.relative_to(fotocasa_originales).as_posix()
    add_file(path.relative_to(PROJECT_ROOT), f"originales/fotocasa/{relative_html_path}")
if not any(path.suffix.lower() == ".html" for path in
           (item["local_path"] for item in files)):
    raise FileNotFoundError("No se encontraron HTML válidos de Fotocasa.")

for filename in (
    "Demandantes-Etxebide-Municipio-2026-04-01_limpio.csv",
    "Demandantes-Etxebide-Municipio-2026-07-01_limpio.csv",
):
    add_file(Path("DatosAlvaro") / "Vivienda" / "VPO" / filename,
             f"originales/etxebide/{filename}")

add_file(
    Path("DatosAlvaro") / "Salario" / "iovj-empleo-economico.csv",
    "originales/otras_fuentes/iovj-empleo-economico.csv",
)
add_file(
    Path("DatosAlvaro") / "Vivienda" / "pisos_euskadi.csv",
    "originales/pisos/pisos_euskadi.csv",
)

missing_files = [str(item["local_path"]) for item in files
                 if not item["local_path"].is_file()]
if missing_files:
    raise FileNotFoundError("Faltan archivos requeridos:\n  " +
                            "\n  ".join(missing_files))

empty_files = [str(item["local_path"]) for item in files
               if item["local_path"].stat().st_size == 0]
if empty_files:
    raise ValueError("Hay archivos vacíos:\n  " + "\n  ".join(empty_files))

for item in files:
    path = item["local_path"]
    if path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8-sig") as source:
            json.load(source)
    elif path.suffix.lower() == ".csv":
        with path.open("r", encoding="utf-8-sig", newline="") as source:
            header = next((row for row in csv.reader(source)
                           if any(cell.strip() for cell in row)), None)
        if not header:
            raise ValueError(f"El CSV no tiene encabezado: {path}")

destinations = [item["hdfs_relative"] for item in files]
if len(destinations) != len(set(destinations)):
    raise ValueError("El manifiesto contiene destinos HDFS duplicados.")

print(f"Archivos originales validados: {len(files)}")
print(f"HTML excluidos por notFound: {len(excluded_html)}")
print(f"Tamaño total: {sum(item['local_path'].stat().st_size for item in files):,} bytes")

Archivos originales validados: 269
HTML excluidos por notFound: 4
Tamaño total: 136,091,377 bytes


## 4. Comprobar conexiones

Se consulta WebHDFS y se comprueba que el tópico configurado para el Reto 1 tenga un líder disponible antes de preparar el consumidor. Si Kafka está terminando de crear/recuperar el líder, se reintenta durante un minuto. Las carpetas HDFS se crean solo cuando Kafka ya está listo; los archivos se suben en las celdas siguientes.

In [4]:
hdfs_client = InsecureClient(HDFS_URL, user=USUARIO)
print("Conexión WebHDFS correcta; carpetas de la raíz:", hdfs_client.list("/"))
producer = Producer({
    "bootstrap.servers": KAFKA_BROKER,
    "client.id": socket.gethostname(),
})
leader_deadline = time.monotonic() + 60
leader_warning_shown = False
while True:
    try:
        metadata = producer.list_topics(topic=TOPIC, timeout=15)
    except KafkaException as error:
        raise RuntimeError(
            f"No se pudo consultar Kafka en {KAFKA_BROKER}; comprueba el broker "
            "y la conectividad de red."
        ) from error

    topic_metadata = metadata.topics.get(TOPIC)
    if topic_metadata is None:
        raise RuntimeError(f"Kafka no devolvió metadatos para el tópico {TOPIC}.")
    if topic_metadata.error is not None:
        topic_error = topic_metadata.error
        retry_leader = topic_error.code() == KafkaError.LEADER_NOT_AVAILABLE
        if not retry_leader:
            raise RuntimeError(
                f"Kafka no puede acceder al tópico {TOPIC}: {topic_error}. "
                "Comprueba el nombre del tópico y los permisos del usuario."
            )
    else:
        if not topic_metadata.partitions:
            raise RuntimeError(f"El tópico Kafka {TOPIC} no tiene particiones.")
        unavailable_partitions = [
            partition_id
            for partition_id, partition in topic_metadata.partitions.items()
            if partition.leader == -1
            or partition.error is not None
        ]
        if not unavailable_partitions:
            break
        retry_leader = True

    if time.monotonic() >= leader_deadline:
        raise TimeoutError(
            f"Kafka sigue sin asignar un líder disponible al tópico {TOPIC} "
            "tras 60 segundos. No se enviaron archivos. Comprueba con el "
            "administrador que el tópico tenga particiones y que sus brokers "
            "estén activos."
        )
    if not leader_warning_shown:
        print(f"El tópico {TOPIC} aún no tiene líder disponible; reintentando...")
        leader_warning_shown = True
    time.sleep(5)

print(f"Kafka listo: {TOPIC}, {len(topic_metadata.partitions)} partición(es).")

consumer = Consumer({
    "bootstrap.servers": KAFKA_BROKER,
    "group.id": GROUP_ID,
    "auto.offset.reset": "latest",
    "enable.auto.commit": False,
})
consumer.subscribe([TOPIC])
assignment_deadline = time.monotonic() + 30
while not consumer.assignment():
    consumer.poll(0.5)
    if time.monotonic() >= assignment_deadline:
        consumer.close()
        raise TimeoutError(
            f"El tópico {TOPIC} existe, pero Kafka no asignó particiones al "
            f"consumidor {GROUP_ID}. Comprueba ACL y errores de grupo del broker."
        )
print("Consumidor Kafka preparado:", consumer.assignment())

HDFS_DIRECTORIES = sorted({
    str(PurePosixPath(item["hdfs_relative"]).parent)
    for item in files
})
for relative_directory in HDFS_DIRECTORIES:
    hdfs_client.makedirs(str(HDFS_ROOT / relative_directory))

print("Conexión Kafka correcta.")

Conexión WebHDFS correcta; carpetas de la raíz: ['amshbase', 'app-logs', 'apps', 'bgtp', 'bigDataTxurdi', 'completed-jobs', 'cursoIABD', 'hadoop', 'mapred', 'mr-history', 'spark-history', 'test', 'tmp', 'user', 'warehouse']
Kafka listo: reto1_archivos_iabd-user11, 1 partición(es).
Consumidor Kafka preparado: [TopicPartition{topic=reto1_archivos_iabd-user11,partition=0,offset=-1000,leader_epoch=None,error=None}]
Conexión Kafka correcta.


## 5. Productor: enviar los archivos por Kafka

Antes de producir mensajes se comprueba cada destino en HDFS. Los archivos que ya existen y tienen el mismo tamaño se omiten; solo se envían los nuevos o los que no coinciden en tamaño. Cada archivo pendiente se divide en mensajes de 128 KiB codificados en Base64.

In [5]:
files_to_upload = []
already_present = []
for item in files:
    remote_path = HDFS_ROOT / PurePosixPath(item["hdfs_relative"])
    remote_status = hdfs_client.status(str(remote_path), strict=False)
    if remote_status is not None and remote_status["type"] != "FILE":
        raise IsADirectoryError(
            f"El destino HDFS existe pero no es un archivo: {remote_path}"
        )
    local_size = item["local_path"].stat().st_size
    if remote_status is not None and remote_status["length"] == local_size:
        already_present.append(item)
    else:
        files_to_upload.append(item)

print(f"Ya estaban en HDFS y se omiten: {len(already_present)}")
print(f"Nuevos o de tamaño distinto para subir: {len(files_to_upload)}")

file_specs = []
for item in files_to_upload:
    size = item["local_path"].stat().st_size
    file_specs.append({
        "path": item["hdfs_relative"],
        "chunks": math.ceil(size / CHUNK_SIZE),
    })
expected_chunks = {item["path"]: item["chunks"] for item in file_specs}
delivery_errors = []

def delivery_report(error, message):
    if error is not None:
        delivery_errors.append(str(error))

def send_event(event, key):
    payload = json.dumps(event, ensure_ascii=False).encode("utf-8")
    while True:
        try:
            producer.produce(TOPIC, key=key.encode("utf-8"), value=payload,
                             callback=delivery_report)
            return
        except BufferError:
            producer.poll(0.5)

send_event({
    "run_id": RUN_ID,
    "type": "start",
    "files": file_specs,
}, RUN_ID)

chunks_sent = 0
for item in files_to_upload:
    relative_path = item["hdfs_relative"]
    with item["local_path"].open("rb") as source:
        chunk_index = 0
        while True:
            chunk = source.read(CHUNK_SIZE)
            if not chunk:
                break
            send_event({
                "run_id": RUN_ID,
                "type": "chunk",
                "path": relative_path,
                "index": chunk_index,
                "data": base64.b64encode(chunk).decode("ascii"),
            }, relative_path)
            chunk_index += 1
            chunks_sent += 1

send_event({"run_id": RUN_ID, "type": "done"}, RUN_ID)
pending = producer.flush(120)
if pending:
    raise TimeoutError(f"Quedaron {pending} mensajes sin confirmar en Kafka.")
if delivery_errors:
    raise RuntimeError("Errores de entrega Kafka: " + "; ".join(delivery_errors))

print(f"Kafka confirmó {chunks_sent} fragmentos para {len(files_to_upload)} archivos.")

Ya estaban en HDFS y se omiten: 0
Nuevos o de tamaño distinto para subir: 269
Kafka confirmó 1091 fragmentos para 269 archivos.


## 6. Consumidor: reconstruir y guardar en HDFS

El consumidor reconstruye cada archivo en un temporal local y, al recibir todos sus fragmentos ordenados, lo sube con WebHDFS a la ruta indicada por el manifiesto. Así evita cargar archivos grandes completos en memoria. Si falta un fragmento, la ejecución termina con error en vez de informar una carga completa.

In [6]:
next_chunk = {path: 0 for path in expected_chunks}
completed_files = set()
start_received = False
done_received = False
last_message_at = time.monotonic()

try:
    with tempfile.TemporaryDirectory(prefix="reto1_hdfs_") as temporary_directory:
        temporary_root = Path(temporary_directory)
        while not (done_received and len(completed_files) == len(expected_chunks)):
            message = consumer.poll(2.0)
            if message is None:
                if time.monotonic() - last_message_at > IDLE_TIMEOUT_SECONDS:
                    raise TimeoutError(
                        "Kafka dejó de entregar mensajes antes de completar "
                        "todos los archivos. Revisa el tópico y los logs."
                    )
                continue
            if message.error():
                raise RuntimeError(f"Error del consumidor Kafka: {message.error()}")

            event = json.loads(message.value().decode("utf-8"))
            if event.get("run_id") != RUN_ID:
                continue
            last_message_at = time.monotonic()

            if event["type"] == "start":
                declared = {entry["path"]: entry["chunks"]
                            for entry in event["files"]}
                if declared != expected_chunks:
                    raise ValueError("El manifiesto recibido por Kafka no coincide.")
                start_received = True
            elif event["type"] == "chunk":
                relative_path = PurePosixPath(event["path"])
                if relative_path.is_absolute() or ".." in relative_path.parts:
                    raise ValueError(f"Ruta de archivo no permitida: {relative_path}")
                path_key = relative_path.as_posix()
                if path_key not in expected_chunks:
                    raise ValueError(f"Archivo no incluido en el manifiesto: {path_key}")
                index = event["index"]
                if index < next_chunk[path_key]:
                    continue
                if index != next_chunk[path_key]:
                    raise ValueError(
                        f"Fragmento fuera de orden para {path_key}: {index}"
                    )
                decoded_chunk = base64.b64decode(event["data"], validate=True)
                local_path = temporary_root.joinpath(*relative_path.parts)
                local_path.parent.mkdir(parents=True, exist_ok=True)
                with local_path.open("ab") as target:
                    target.write(decoded_chunk)
                next_chunk[path_key] += 1

                if next_chunk[path_key] == expected_chunks[path_key]:
                    remote_path = HDFS_ROOT / relative_path
                    hdfs_client.makedirs(str(remote_path.parent))
                    hdfs_client.upload(str(remote_path), str(local_path), overwrite=True)
                    if hdfs_client.status(str(remote_path), strict=False) is None:
                        raise IOError(f"No se encontró el archivo en HDFS: {remote_path}")
                    completed_files.add(path_key)
                    print(f"Subido: {remote_path}")
            elif event["type"] == "done":
                done_received = True
finally:
    consumer.close()

if not start_received:
    raise RuntimeError("No se recibió el manifiesto de esta ejecución.")
if completed_files != set(expected_chunks):
    missing = sorted(set(expected_chunks) - completed_files)
    raise RuntimeError("Archivos no completados: " + ", ".join(missing))
print(
    f"Carga completa: {len(completed_files)} archivos subidos, "
    f"{len(already_present)} ya existentes omitidos, en {HDFS_ROOT}."
)

Subido: /user/iabd-user11/reto1/originales/fotocasa/MunicipiosCAE_2025_19_02.xlsx
Subido: /user/iabd-user11/reto1/originales/emal/EMAL.-Barrios-Municipios.-2016-2026_es.xlsx
Subido: /user/iabd-user11/reto1/originales/eustat/eustat_distribucion_renta_municipal.json
Subido: /user/iabd-user11/reto1/originales/eustat/eustat_distribucion_renta_por_edad.json
Subido: /user/iabd-user11/reto1/originales/eustat/eustat_indicadores_municipales.json
Subido: /user/iabd-user11/reto1/originales/eustat/eustat_percentiles_renta_municipales.json
Subido: /user/iabd-user11/reto1/originales/eustat/eustat_renta_por_edad.json
Subido: /user/iabd-user11/reto1/originales/eustat/eustat_renta_por_municipio.json
Subido: /user/iabd-user11/reto1/originales/eustat/metadatos_distribucion_renta_municipal.json
Subido: /user/iabd-user11/reto1/originales/eustat/metadatos_distribucion_renta_por_edad.json
Subido: /user/iabd-user11/reto1/originales/eustat/metadatos_indicadores_municipales.json
Subido: /user/iabd-user11/reto1/

## 7. Verificación final

Se comprueba que todas las rutas del manifiesto existen en HDFS. La estructura de carpetas resultante queda bajo `HDFS_ROOT/originales/`, organizada por fuente.

In [7]:
missing_in_hdfs = []
for item in files:
    remote_path = HDFS_ROOT / PurePosixPath(item["hdfs_relative"])
    if hdfs_client.status(str(remote_path), strict=False) is None:
        missing_in_hdfs.append(str(remote_path))

if missing_in_hdfs:
    raise FileNotFoundError(
        "No aparecen en HDFS los siguientes archivos:\n  " +
        "\n  ".join(missing_in_hdfs)
    )
print(f"Verificación correcta: {len(files)} archivos comprobados en HDFS.")
print("Contenido de la raíz HDFS:", hdfs_client.list(str(HDFS_ROOT)))

Verificación correcta: 269 archivos comprobados en HDFS.
Contenido de la raíz HDFS: ['originales']
